In [13]:
# =============================================================================
# CELL 0: Imports
# =============================================================================

import pandas as pd
import openpyxl
import numpy as np
from pathlib import Path
from datetime import datetime
import warnings

In [2]:
# =============================================================================
# CELL 1: Configuration and Setup (Gas Prices Data)
# =============================================================================

NOTEBOOK_NAME = "05: Gas Prices Data Ingestion"
DATA_SUBFOLDER = "gas_prices"
OBJECTIVE = """Process Austrian Energy Agency gas price data (OEGPI)
Monthly gas prices with 4 time horizons: month, quarter, season, year
Output: Monthly aggregated data only

IMPORTANT ARCHITECTURAL DECISION:
Gas price data is MONTHLY ONLY - no daily or weekly rows will be created.
Consistent with climate and production data architecture.
Gas price variables can only be used in monthly-level analyses.

DATE FORMAT: YYYY-Mmm (e.g., 2020-Jan) with German month abbreviations
Requires custom parsing with German month mapping."""

# Gas prices specific configuration
GAS_FILE_NAME = "oegpi_data.xlsx"
DATE_COLUMN = "Datum"
SKIPROWS = list(range(10))  # Skip rows 0-9 (metadata and jpg)

# German month abbreviations mapping
GERMAN_MONTHS = {
    'Jan': '01', 'Feb': '02', 'Mrz': '03', 'Apr': '04',
    'Mai': '05', 'Jun': '06', 'Jul': '07', 'Aug': '08',
    'Sep': '09', 'Okt': '10', 'Nov': '11', 'Dez': '12'
}

# Column mapping: Source column name → Target column name
COLUMN_MAPPING = {
    'Datum': 'date',
    'Monat': 'oegpi_month',
    'Quartal': 'oegpi_quarter',
    'Saison': 'oegpi_season',
    'Jahr': 'oegpi_year'
}

# Setup functions (reused from previous pipelines - consistent)
def setup_pandas_options():
    """Configure pandas display options for better data inspection."""
    pd.set_option('display.max_columns', None)
    pd.set_option('display.max_rows', 100)  
    pd.set_option('display.width', None)
    warnings.filterwarnings('ignore')

def setup_project_paths(data_subfolder):
    """Set up project directory paths for any data source."""
    PROJECT_ROOT = Path.cwd().parent if 'notebooks' in str(Path.cwd()) else Path.cwd()
    RAW_DATA_PATH = PROJECT_ROOT / 'data' / 'raw' / data_subfolder
    PROCESSED_DATA_PATH = PROJECT_ROOT / 'data' / 'processed'
    PROCESSED_DATA_PATH.mkdir(parents=True, exist_ok=True)
    return PROJECT_ROOT, RAW_DATA_PATH, PROCESSED_DATA_PATH

def print_notebook_header(notebook_name, objective, raw_path, processed_path):
    """Print standardized notebook start information."""
    print("="*60)
    print(f"NOTEBOOK: {notebook_name}")
    print("="*60)
    print(f"Start Time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"Raw Data Path: {raw_path}")
    print(f"Processed Data Path: {processed_path}")
    print(f"Objective: {objective}")
    print("="*60)

# Setup execution
setup_pandas_options()
PROJECT_ROOT, RAW_DATA_PATH, PROCESSED_DATA_PATH = setup_project_paths(DATA_SUBFOLDER)
print_notebook_header(NOTEBOOK_NAME, OBJECTIVE, RAW_DATA_PATH, PROCESSED_DATA_PATH)

print("Cell 1: Configuration and Setup - READY")

NOTEBOOK: 05: Gas Prices Data Ingestion
Start Time: 2025-10-01 14:08:51
Raw Data Path: c:\Users\paulr\Desktop\Capstone\data\raw\gas_prices
Processed Data Path: c:\Users\paulr\Desktop\Capstone\data\processed
Objective: Process Austrian Energy Agency gas price data (OEGPI)
Monthly gas prices with 4 time horizons: month, quarter, season, year
Output: Monthly aggregated data only

IMPORTANT ARCHITECTURAL DECISION:
Gas price data is MONTHLY ONLY - no daily or weekly rows will be created.
Consistent with climate and production data architecture.
Gas price variables can only be used in monthly-level analyses.

DATE FORMAT: YYYY-Mmm (e.g., 2020-Jan) with German month abbreviations
Requires custom parsing with German month mapping.
Cell 1: Configuration and Setup - READY


In [3]:
# =============================================================================
# CELL 2: Missing Values Standardization Function (with Quality Control)
# =============================================================================

def standardize_missing_values(df, additional_missing=None, show_quality_control=True):
    """
    Convert various missing value representations to pandas NaN.
    
    Args:
        df (DataFrame): Input dataframe
        additional_missing (list): Extra missing indicators beyond defaults
        show_quality_control (bool): Whether to display quality control output
    
    Returns:
        DataFrame: Dataframe with standardized missing values
        dict: Report of found missing patterns
    """
    missing_indicators = [
        'N/A', 'n/a', 'NA', 'na',
        '', '-', '--', '---',
        'NULL', 'null', 'Null',
        'NaN', 'nan', '#N/A'
    ]
    
    if additional_missing:
        missing_indicators.extend(additional_missing)
    
    if show_quality_control:
        print("\nMISSING VALUES STANDARDIZATION - QUALITY CONTROL:")
        print("-" * 55)
        print("Missing indicators standardized:")
        print("  ", ', '.join([f"'{ind}'" for ind in missing_indicators]))
        print()
    
    found_patterns = {}
    df_clean = df.copy()
    
    for col in df_clean.columns:
        original_nulls = df_clean[col].isnull().sum()
        df_clean[col] = df_clean[col].replace(missing_indicators, np.nan)
        new_nulls = df_clean[col].isnull().sum()
        converted_count = new_nulls - original_nulls
        
        if converted_count > 0:
            found_patterns[col] = {
                'original_nulls': original_nulls,
                'converted_missing': converted_count,
                'total_nulls': new_nulls
            }
    
    if show_quality_control:
        if found_patterns:
            print("CONVERSION RESULTS BY COLUMN:")
            for col, pattern in found_patterns.items():
                print(f"  {col}:")
                print(f"    Original nulls: {pattern['original_nulls']}")
                print(f"    Converted missing: {pattern['converted_missing']}")
                print(f"    Total nulls: {pattern['total_nulls']}")
        else:
            print("No missing value patterns found for conversion")
        print("-" * 55)
    
    return df_clean, found_patterns

print("Cell 2: Missing Values Standardization Function - READY")

Cell 2: Missing Values Standardization Function - READY


In [4]:
# =============================================================================
# CELL 3: Gas Prices File Discovery
# =============================================================================

def discover_gas_file(data_path, filename):
    """
    Discover single gas prices data file.
    
    Args:
        data_path (Path): Path to raw data directory
        filename (str): Expected filename
    
    Returns:
        dict: File information dictionary
    """
    file_path = data_path / filename
    
    if file_path.exists():
        return {
            'filename': filename,
            'path': str(file_path),
            'exists': True
        }
    else:
        return {
            'filename': filename,
            'path': str(file_path),
            'exists': False
        }

def display_file_discovery_results(file_info):
    """Display results of gas prices file discovery."""
    print("Gas prices file discovery:")
    print(f"  File: {file_info['filename']}")
    print(f"  Exists: {file_info['exists']}")
    print(f"  Path: {file_info['path']}")
    
    if file_info['exists']:
        print("Ready to proceed with gas prices data loading")
    else:
        print("ERROR: Gas prices file not found!")

# Execute file discovery
gas_file_info = discover_gas_file(RAW_DATA_PATH, GAS_FILE_NAME)
display_file_discovery_results(gas_file_info)

print("Cell 3: Gas Prices File Discovery - COMPLETE")

Gas prices file discovery:
  File: oegpi_data.xlsx
  Exists: True
  Path: c:\Users\paulr\Desktop\Capstone\data\raw\gas_prices\oegpi_data.xlsx
Ready to proceed with gas prices data loading
Cell 3: Gas Prices File Discovery - COMPLETE


In [6]:
# =============================================================================
# CELL 4: Load and Clean Gas Prices Data
# =============================================================================

def parse_german_date(date_str):
    """
    Parse German date format YYYY-Mmm to datetime.
    
    Args:
        date_str (str): Date string in format "2020-Jan"
    
    Returns:
        datetime: Parsed datetime object (first day of month)
    """
    if pd.isna(date_str):
        return pd.NaT
    
    try:
        # Split year and month abbreviation
        year, month_abbr = str(date_str).split('-')
        
        # Lookup month number from German abbreviation
        month_num = GERMAN_MONTHS.get(month_abbr)
        
        if month_num is None:
            print(f"WARNING: Unknown month abbreviation '{month_abbr}' in date '{date_str}'")
            return pd.NaT
        
        # Construct datetime (first day of month)
        return pd.to_datetime(f"{year}-{month_num}-01")
        
    except Exception as e:
        print(f"ERROR parsing date '{date_str}': {e}")
        return pd.NaT

def load_gas_data(file_path, skiprows, column_mapping):
    """
    Load gas prices data from Excel and select relevant columns.
    
    Args:
        file_path (str/Path): Path to gas prices Excel file
        skiprows (list): Row indices to skip (metadata rows)
        column_mapping (dict): Mapping of source to target column names
    
    Returns:
        tuple: (dataframe, metadata)
    """
    try:
        # Load Excel file with header skip
        df = pd.read_excel(
            file_path,
            skiprows=skiprows
        )
        # Strip whitespace from column names (Excel often has trailing spaces)
        df.columns = df.columns.str.strip()

        print(f"INITIAL DATA LOAD:")
        print(f"  Original shape: {df.shape}")
        print(f"  Rows after header skip: {len(df)}")
        print(f"  Total columns available: {len(df.columns)}")
        print(f"  Column names: {df.columns.tolist()}")
        print()
        
        # Select only relevant columns (date + 4 price columns)
        source_columns = list(column_mapping.keys())
        
        print(f"COLUMN SELECTION:")
        print(f"  Selecting {len(source_columns)} of {len(df.columns)} columns")
        print(f"  Source columns: {source_columns}")
        print()
        
        # Column selection
        df_filtered = df[source_columns].copy()
        
        print(f"After column selection: {df_filtered.shape}")
        print()
        
        # Parse German dates BEFORE cleaning missing values
        print("DATE PARSING:")
        print(f"  Parsing German date format (YYYY-Mmm)")
        print(f"  Sample dates before parsing: {df_filtered[DATE_COLUMN].head(3).tolist()}")
        
        df_filtered[DATE_COLUMN] = df_filtered[DATE_COLUMN].apply(parse_german_date)
        
        print(f"  Sample dates after parsing: {df_filtered[DATE_COLUMN].head(3).tolist()}")
        print()
        
        # Clean missing values BEFORE renaming
        df_clean, missing_patterns = standardize_missing_values(df_filtered, show_quality_control=True)
        
        # Rename columns to standardized names
        df_clean.rename(columns=column_mapping, inplace=True)
        
        print(f"\nCOLUMN RENAMING:")
        print(f"  Renamed {len(column_mapping)} columns with oegpi_ prefix")
        print(f"  New columns: {df_clean.columns.tolist()}")
        print()
        
        # Create metadata
        metadata = {
            'filename': Path(file_path).name,
            'rows': len(df_clean),
            'columns': list(df_clean.columns),
            'missing_patterns_found': missing_patterns,
            'date_range': (df_clean['date'].min(), df_clean['date'].max()) if len(df_clean) > 0 else (None, None)
        }
        
        print(f"DATA SUMMARY:")
        print(f"  Final shape: {df_clean.shape}")
        print(f"  Date range: {metadata['date_range'][0].strftime('%Y-%m-%d')} to {metadata['date_range'][1].strftime('%Y-%m-%d')}")
        print()
        
        print("SAMPLE DATA:")
        print(df_clean.head(10))
        
        return df_clean, metadata
        
    except Exception as e:
        print(f"ERROR loading gas data: {e}")
        return None, {'error': str(e)}

# Execute data loading
if gas_file_info['exists']:
    gas_df, gas_metadata = load_gas_data(
        gas_file_info['path'],
        SKIPROWS,
        COLUMN_MAPPING
    )
    
    if gas_df is None:
        print(f"ERROR loading gas data: {gas_metadata['error']}")
else:
    print("Skipping data load - file not found")

print("Cell 4: Load and Clean Gas Prices Data - COMPLETE")

INITIAL DATA LOAD:
  Original shape: (69, 5)
  Rows after header skip: 69
  Total columns available: 5
  Column names: ['Datum', 'Monat', 'Quartal', 'Saison', 'Jahr']

COLUMN SELECTION:
  Selecting 5 of 5 columns
  Source columns: ['Datum', 'Monat', 'Quartal', 'Saison', 'Jahr']

After column selection: (69, 5)

DATE PARSING:
  Parsing German date format (YYYY-Mmm)
  Sample dates before parsing: ['2020-Jan', '2020-Feb', '2020-Mrz']
  Sample dates after parsing: [Timestamp('2020-01-01 00:00:00'), Timestamp('2020-02-01 00:00:00'), Timestamp('2020-03-01 00:00:00')]


MISSING VALUES STANDARDIZATION - QUALITY CONTROL:
-------------------------------------------------------
Missing indicators standardized:
   'N/A', 'n/a', 'NA', 'na', '', '-', '--', '---', 'NULL', 'null', 'Null', 'NaN', 'nan', '#N/A'

No missing value patterns found for conversion
-------------------------------------------------------

COLUMN RENAMING:
  Renamed 5 columns with oegpi_ prefix
  New columns: ['date', 'oegpi_mon

In [8]:
# =============================================================================
# CELL 5: Convert Data Types and Create Long Format
# =============================================================================

def convert_gas_data_types(df):
    """
    Convert gas price columns to appropriate numeric types.
    Gas prices are kept as float64 (decimals needed for price precision).
    
    Args:
        df (DataFrame): Gas data with oegpi_ prefixed columns
    
    Returns:
        DataFrame: Data with converted types
    """
    df_converted = df.copy()
    
    print("DATA TYPE CONVERSION:")
    print("-" * 40)
    
    # Get all gas price columns (exclude date column)
    gas_columns = [col for col in df_converted.columns if col.startswith('oegpi_')]
    
    print(f"Converting {len(gas_columns)} gas price columns to numeric:")
    
    for col in gas_columns:
        # Convert to numeric (handles any string values)
        df_converted[col] = pd.to_numeric(df_converted[col], errors='coerce')
        
        # Keep as float64 (prices need decimal precision)
        df_converted[col] = df_converted[col].astype('float64')
        
        print(f"  {col}: float64")
    
    print()
    print("NOTE: Gas prices kept as float64 for decimal precision and because this data is publicly available and needs no coarsening to be published..")
    
    return df_converted

def transform_gas_to_long_format(df, date_column='date'):
    """
    Transform gas price data to long format.
    IMPORTANT: Only creates MONTHLY rows (no daily/weekly as per architecture decision).
    
    Args:
        df (DataFrame): Gas price data with standardized columns
        date_column (str): Name of date column
    
    Returns:
        DataFrame: Gas price data in long format (monthly only)
    """
    print("\nTRANSFORMING TO LONG FORMAT:")
    print("-" * 40)
    print("ARCHITECTURAL NOTE: Creating MONTHLY rows only")
    print("No daily or weekly rows will be created for gas price data")
    print("Consistent with climate and production data architecture")
    print()
    
    df_long = df.copy()
    
    # Ensure date is datetime
    df_long[date_column] = pd.to_datetime(df_long[date_column])
    
    # Add time components
    df_long['year'] = df_long[date_column].dt.year
    df_long['month'] = df_long[date_column].dt.month
    df_long['quarter'] = df_long[date_column].dt.quarter
    df_long['week'] = df_long[date_column].dt.isocalendar().week
    df_long['month_name'] = df_long[date_column].dt.month_name()
    
    # Add aggregation level - MONTHLY ONLY
    df_long['aggregation_level'] = 'monthly'
    
    # Convert date to first of month in ISO format (YYYY-MM-DD)
    df_long['date'] = df_long[date_column].apply(lambda x: x.replace(day=1).strftime('%Y-%m-%d'))
    
    # Select final columns in correct order
    base_columns = ['date', 'year', 'month', 'quarter', 'week', 'aggregation_level', 'month_name']
    gas_columns = [col for col in df_long.columns if col.startswith('oegpi_')]
    final_columns = base_columns + gas_columns
    
    df_final = df_long[final_columns].copy()
    
    print(f"FINAL STRUCTURE:")
    print(f"  Rows: {len(df_final)} (monthly only)")
    print(f"  Columns: {len(df_final.columns)}")
    print(f"  Gas price variables: {len(gas_columns)}")
    print(f"  Date range: {df_final['date'].min()} to {df_final['date'].max()}")
    print()
    
    print("Gas price columns:")
    for col in gas_columns:
        print(f"  {col}")
    print()
    
    print("Sample data:")
    print(df_final.head())
    
    return df_final

# Execute data type conversion and transformation
if 'gas_df' in locals() and gas_df is not None:
    gas_converted = convert_gas_data_types(gas_df)
    
    print(f"\nData types after conversion:")
    for col in gas_converted.columns:
        if col.startswith('oegpi_'):
            print(f"  {col}: {gas_converted[col].dtype}")
    
    gas_long_format = transform_gas_to_long_format(gas_converted)
    
    print("\nLONG FORMAT TRANSFORMATION SUCCESSFUL")
    print(f"  Shape: {gas_long_format.shape}")
else:
    print("Skipping transformation - gas data not available")

print("Cell 5: Convert Data Types and Create Long Format - COMPLETE")

DATA TYPE CONVERSION:
----------------------------------------
Converting 4 gas price columns to numeric:
  oegpi_month: float64
  oegpi_quarter: float64
  oegpi_season: float64
  oegpi_year: float64

NOTE: Gas prices kept as float64 for decimal precision and because this data is publicly available and needs no coarsening to be published..

Data types after conversion:
  oegpi_month: float64
  oegpi_quarter: float64
  oegpi_season: float64
  oegpi_year: float64

TRANSFORMING TO LONG FORMAT:
----------------------------------------
ARCHITECTURAL NOTE: Creating MONTHLY rows only
No daily or weekly rows will be created for gas price data
Consistent with climate and production data architecture

FINAL STRUCTURE:
  Rows: 69 (monthly only)
  Columns: 11
  Gas price variables: 4
  Date range: 2020-01-01 to 2025-09-01

Gas price columns:
  oegpi_month
  oegpi_quarter
  oegpi_season
  oegpi_year

Sample data:
         date  year  month  quarter  week aggregation_level month_name  \
0  2020-01-0

In [9]:
# =============================================================================
# CELL 6: Save Gas Prices Dataset
# =============================================================================

def save_gas_dataset(df, output_dir, filename="gas_consolidated.csv"):
    """
    Save gas prices dataset as separate validation sample.
    
    Args:
        df (DataFrame): Final gas prices dataset
        output_dir (Path): Directory for outputs
        filename (str): Output filename
    
    Returns:
        Path: Path to saved file
    """
    if df is None or len(df) == 0:
        print("No gas data to save!")
        return None
    
    output_path = output_dir / filename
    df.to_csv(output_path, index=False)
    
    print(f"GAS PRICES DATASET SAVED:")
    print(f"  Path: {output_path}")
    print(f"  Rows: {len(df)}")
    print(f"  Columns: {len(df.columns)}")
    print(f"  Aggregation level: {df['aggregation_level'].unique()}")
    print(f"  Date range: {df['date'].min()} to {df['date'].max()}")
    
    # Show gas price columns summary
    gas_columns = [col for col in df.columns if col.startswith('oegpi_')]
    print(f"  Gas price variables: {len(gas_columns)}")
    
    return output_path

# Save gas dataset as validation sample
if 'gas_long_format' in locals() and gas_long_format is not None:
    gas_file_path = save_gas_dataset(gas_long_format, PROCESSED_DATA_PATH)
    
    if gas_file_path:
        print("\nGAS PRICES FORK SUCCESSFUL")
        print("Ready for merge with data_consolidated.csv")
else:
    print("Skipping save - gas_long_format not available")

print("Cell 6: Save Gas Prices Dataset - COMPLETE")

GAS PRICES DATASET SAVED:
  Path: c:\Users\paulr\Desktop\Capstone\data\processed\gas_consolidated.csv
  Rows: 69
  Columns: 11
  Aggregation level: ['monthly']
  Date range: 2020-01-01 to 2025-09-01
  Gas price variables: 4

GAS PRICES FORK SUCCESSFUL
Ready for merge with data_consolidated.csv
Cell 6: Save Gas Prices Dataset - COMPLETE


In [10]:
# =============================================================================
# CELL 7: Merge Gas Prices with Consolidated Data
# =============================================================================

def merge_gas_with_consolidated_data(gas_df, consolidated_file_path, output_dir):
    """
    Merge gas price data with existing consolidated data.
    
    Args:
        gas_df (DataFrame): Gas price dataset in long format (monthly only)
        consolidated_file_path (str/Path): Path to data_consolidated.csv
        output_dir (Path): Directory for final output
    
    Returns:
        DataFrame: Merged dataset with gas price data added
    """
    consolidated_path = Path(consolidated_file_path)
    
    if not consolidated_path.exists():
        print(f"ERROR: Consolidated file not found at {consolidated_path}")
        return pd.DataFrame()
    
    print("MERGING GAS PRICES WITH CONSOLIDATED DATA:")
    print("-" * 50)
    
    try:
        # Load existing consolidated data
        consolidated_df = pd.read_csv(consolidated_path)
        
        print(f"Existing consolidated data loaded:")
        print(f"  Shape: {consolidated_df.shape}")
        print(f"  Aggregation levels: {consolidated_df['aggregation_level'].value_counts().to_dict()}")
        print()
        
        print(f"Gas price data to merge:")
        print(f"  Shape: {gas_df.shape}")
        print(f"  Aggregation levels: {gas_df['aggregation_level'].value_counts().to_dict()}")
        print(f"  Date range: {gas_df['date'].min()} to {gas_df['date'].max()}")
        print()
        
        # Merge on date and aggregation_level
        merged_df = pd.merge(
            consolidated_df, 
            gas_df, 
            on=['date', 'aggregation_level'], 
            how='outer',  # Keep all rows from both datasets
            suffixes=('', '_gas')
        )
        
        print(f"After merge:")
        print(f"  Shape: {merged_df.shape}")
        print()
        
        # Handle duplicate columns from merge
        duplicate_cols = ['year', 'month', 'quarter', 'week', 'month_name']
        for col in duplicate_cols:
            gas_col = f"{col}_gas"
            if gas_col in merged_df.columns:
                merged_df[col] = merged_df[col].fillna(merged_df[gas_col])
                merged_df.drop(gas_col, axis=1, inplace=True)
                print(f"  Resolved duplicate: {col}")
        
        # Sort by date and aggregation level
        merged_df = merged_df.sort_values(['date', 'aggregation_level']).reset_index(drop=True)
        
        # Save merged dataset
        final_path = output_dir / "data_consolidated.csv"
        merged_df.to_csv(final_path, index=False)
        
        print(f"\nFINAL CONSOLIDATED DATASET:")
        print(f"  Saved to: {final_path}")
        print(f"  Final shape: {merged_df.shape}")
        print(f"  Date range: {merged_df['date'].min()} to {merged_df['date'].max()}")
        print(f"  Aggregation levels: {merged_df['aggregation_level'].value_counts().to_dict()}")
        print()
        
        # Show which columns are gas-specific
        gas_columns = [col for col in merged_df.columns if col.startswith('oegpi_')]
        print(f"Gas price columns added ({len(gas_columns)}):")
        for col in gas_columns:
            non_null = merged_df[col].notna().sum()
            print(f"  {col}: {non_null} non-null values")
        
        return merged_df
        
    except Exception as e:
        print(f"ERROR during merge: {e}")
        return pd.DataFrame()

# Execute merge with consolidated data
if 'gas_long_format' in locals() and gas_long_format is not None:
    consolidated_file = PROCESSED_DATA_PATH / "data_consolidated.csv"
    final_merged_df = merge_gas_with_consolidated_data(
        gas_long_format, 
        consolidated_file, 
        PROCESSED_DATA_PATH
    )
    
    if len(final_merged_df) > 0:
        print("\nGAS PRICES MERGE SUCCESSFUL")
        print(f"data_consolidated.csv now contains: Electricity + Carbon + Climate + Production + Gas")
    else:
        print("\nMERGE FAILED")
else:
    print("Skipping merge - gas_long_format not available")

print("Cell 7: Merge Gas Prices with Consolidated Data - COMPLETE")

MERGING GAS PRICES WITH CONSOLIDATED DATA:
--------------------------------------------------
Existing consolidated data loaded:
  Shape: (4650, 37)
  Aggregation levels: {'daily': 3896, 'weekly': 566, 'monthly': 188}

Gas price data to merge:
  Shape: (69, 11)
  Aggregation levels: {'monthly': 69}
  Date range: 2020-01-01 to 2025-09-01

After merge:
  Shape: (4651, 46)

  Resolved duplicate: year
  Resolved duplicate: month
  Resolved duplicate: quarter
  Resolved duplicate: week
  Resolved duplicate: month_name

FINAL CONSOLIDATED DATASET:
  Saved to: c:\Users\paulr\Desktop\Capstone\data\processed\data_consolidated.csv
  Final shape: (4651, 41)
  Date range: 2010-01-01 to 2025-09-01
  Aggregation levels: {'daily': 3896, 'weekly': 566, 'monthly': 189}

Gas price columns added (4):
  oegpi_month: 69 non-null values
  oegpi_quarter: 23 non-null values
  oegpi_season: 11 non-null values
  oegpi_year: 23 non-null values

GAS PRICES MERGE SUCCESSFUL
data_consolidated.csv now contains: Elec

In [14]:
# =============================================================================
# CELL 8: Comprehensive Diagnostics for Final Consolidated Dataset
# =============================================================================

def run_comprehensive_diagnostics(consolidated_file_path):
    """
    Run comprehensive quality control diagnostics on final consolidated dataset.
    Verifies data integrity, completeness, and structure across all data sources.
    
    Args:
        consolidated_file_path (Path): Path to data_consolidated.csv
    
    Returns:
        DataFrame: Loaded consolidated dataset for further inspection
    """
    consolidated_path = Path(consolidated_file_path)
    
    if not consolidated_path.exists():
        print(f"ERROR: Consolidated file not found at {consolidated_path}")
        return None
    
    print("="*70)
    print("COMPREHENSIVE DIAGNOSTICS: data_consolidated.csv")
    print("="*70)
    print()
    
    try:
        df = pd.read_csv(consolidated_path)
        
        # =====================================================================
        # 1. OVERALL STRUCTURE
        # =====================================================================
        print("1. OVERALL STRUCTURE:")
        print("-" * 70)
        print(f"  Shape: {df.shape[0]} rows × {df.shape[1]} columns")
        print(f"  Date range (overall): {df['date'].min()} to {df['date'].max()}")
        print(f"  Memory usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
        print()
        
        # =====================================================================
        # 2. AGGREGATION LEVEL DISTRIBUTION
        # =====================================================================
        print("2. AGGREGATION LEVEL DISTRIBUTION:")
        print("-" * 70)
        agg_counts = df['aggregation_level'].value_counts().sort_index()
        for level, count in agg_counts.items():
            pct = (count / len(df)) * 100
            date_range_level = df[df['aggregation_level'] == level]['date'].agg(['min', 'max'])
            print(f"  {level:10s}: {count:5d} rows ({pct:5.1f}%) | Range: {date_range_level['min']} to {date_range_level['max']}")
        print()
        
        # =====================================================================
        # 3. COLUMN INVENTORY BY DATA SOURCE
        # =====================================================================
        print("3. COLUMN INVENTORY BY DATA SOURCE:")
        print("-" * 70)
        
        base_cols = ['date', 'year', 'month', 'quarter', 'week', 'aggregation_level', 'month_name']
        electricity_cols = [col for col in df.columns if col.startswith('price_') or col == 'data_completeness']
        carbon_cols = [col for col in df.columns if col.startswith('carbonprices_')]
        climate_cols = [col for col in df.columns if col.startswith('climate_')]
        production_cols = [col for col in df.columns if col.startswith('prod_')]
        gas_cols = [col for col in df.columns if col.startswith('oegpi_')]
        
        print(f"  Base structure: {len(base_cols)} columns")
        print(f"  Electricity:    {len(electricity_cols)} columns - {electricity_cols}")
        print(f"  Carbon:         {len(carbon_cols)} columns - {carbon_cols}")
        print(f"  Climate:        {len(climate_cols)} columns - {climate_cols}")
        print(f"  Production:     {len(production_cols)} columns")
        print(f"    (Showing first 5: {production_cols[:5]})")
        print(f"  Gas Prices:     {len(gas_cols)} columns - {gas_cols}")
        print(f"  Total: {len(df.columns)} columns")
        print()
        
        # =====================================================================
        # 4. MISSING VALUES ANALYSIS
        # =====================================================================
        print("4. MISSING VALUES ANALYSIS:")
        print("-" * 70)
        
        def analyze_missing_by_source(columns, source_name):
            if len(columns) == 0:
                print(f"  {source_name}: No columns found")
                return
            
            print(f"  {source_name}:")
            for col in columns:
                missing_count = df[col].isna().sum()
                missing_pct = (missing_count / len(df)) * 100
                print(f"    {col:45s}: {missing_count:5d} missing ({missing_pct:5.1f}%)")
        
        analyze_missing_by_source(electricity_cols, "ELECTRICITY")
        print()
        analyze_missing_by_source(carbon_cols, "CARBON")
        print()
        analyze_missing_by_source(climate_cols, "CLIMATE")
        print()
        
        # Production columns - show summary
        if len(production_cols) > 0:
            print(f"  PRODUCTION (summary of {len(production_cols)} columns):")
            for col in production_cols:
                missing_count = df[col].isna().sum()
                missing_pct = (missing_count / len(df)) * 100
                print(f"    {col:45s}: {missing_count:5d} missing ({missing_pct:5.1f}%)")
        print()
        
        analyze_missing_by_source(gas_cols, "GAS PRICES")
        print()
        
        # =====================================================================
        # 5. DATA TYPES VERIFICATION
        # =====================================================================
        print("5. DATA TYPES VERIFICATION:")
        print("-" * 70)
        print("  Expected: All numeric columns should be float64 (due to CSV format)")
        print()
        
        numeric_cols = electricity_cols + carbon_cols + climate_cols + production_cols + gas_cols
        non_float_cols = [col for col in numeric_cols if df[col].dtype != 'float64']
        
        if len(non_float_cols) > 0:
            print(f"  WARNING: {len(non_float_cols)} columns are not float64:")
            for col in non_float_cols:
                print(f"    {col}: {df[col].dtype}")
        else:
            print(f"  ✓ All {len(numeric_cols)} numeric columns are float64")
        print()
        
        # =====================================================================
        # 6. ARCHITECTURAL VALIDATION
        # =====================================================================
        print("6. ARCHITECTURAL VALIDATION:")
        print("-" * 70)
        
        # Check daily rows - should only have electricity + carbon
        daily_df = df[df['aggregation_level'] == 'daily']
        if len(daily_df) > 0:
            daily_climate_nulls = daily_df[climate_cols].isna().all(axis=1).sum()
            daily_prod_nulls = daily_df[production_cols].isna().all(axis=1).sum() if len(production_cols) > 0 else len(daily_df)
            daily_gas_nulls = daily_df[gas_cols].isna().all(axis=1).sum() if len(gas_cols) > 0 else len(daily_df)
            print(f"  Daily rows ({len(daily_df)}):")
            print(f"    Climate columns all null: {daily_climate_nulls}/{len(daily_df)} rows ({daily_climate_nulls/len(daily_df)*100:.1f}%) ✓")
            print(f"    Production columns all null: {daily_prod_nulls}/{len(daily_df)} rows ({daily_prod_nulls/len(daily_df)*100:.1f}%) ✓")
            print(f"    Gas columns all null: {daily_gas_nulls}/{len(daily_df)} rows ({daily_gas_nulls/len(daily_df)*100:.1f}%) ✓")
        
        # Check weekly rows - should only have electricity + carbon
        weekly_df = df[df['aggregation_level'] == 'weekly']
        if len(weekly_df) > 0:
            weekly_climate_nulls = weekly_df[climate_cols].isna().all(axis=1).sum()
            weekly_prod_nulls = weekly_df[production_cols].isna().all(axis=1).sum() if len(production_cols) > 0 else len(weekly_df)
            weekly_gas_nulls = weekly_df[gas_cols].isna().all(axis=1).sum() if len(gas_cols) > 0 else len(weekly_df)
            print(f"  Weekly rows ({len(weekly_df)}):")
            print(f"    Climate columns all null: {weekly_climate_nulls}/{len(weekly_df)} rows ({weekly_climate_nulls/len(weekly_df)*100:.1f}%) ✓")
            print(f"    Production columns all null: {weekly_prod_nulls}/{len(weekly_df)} rows ({weekly_prod_nulls/len(weekly_df)*100:.1f}%) ✓")
            print(f"    Gas columns all null: {weekly_gas_nulls}/{len(weekly_df)} rows ({weekly_gas_nulls/len(weekly_df)*100:.1f}%) ✓")
        
        # Check monthly rows - should have all sources
        monthly_df = df[df['aggregation_level'] == 'monthly']
        if len(monthly_df) > 0:
            monthly_climate_data = monthly_df[climate_cols].notna().any(axis=1).sum()
            monthly_prod_data = monthly_df[production_cols].notna().any(axis=1).sum() if len(production_cols) > 0 else 0
            monthly_gas_data = monthly_df[gas_cols].notna().any(axis=1).sum() if len(gas_cols) > 0 else 0
            print(f"  Monthly rows ({len(monthly_df)}):")
            print(f"    Rows with climate data: {monthly_climate_data}/{len(monthly_df)} ({monthly_climate_data/len(monthly_df)*100:.1f}%)")
            print(f"    Rows with production data: {monthly_prod_data}/{len(monthly_df)} ({monthly_prod_data/len(monthly_df)*100:.1f}%)")
            print(f"    Rows with gas data: {monthly_gas_data}/{len(monthly_df)} ({monthly_gas_data/len(monthly_df)*100:.1f}%)")
        print()
        
        # =====================================================================
        # 7. SAMPLE DATA FROM EACH AGGREGATION LEVEL
        # =====================================================================
        print("7. SAMPLE DATA FROM EACH AGGREGATION LEVEL:")
        print("-" * 70)
        
        for level in ['daily', 'weekly', 'monthly']:
            level_df = df[df['aggregation_level'] == level]
            if len(level_df) > 0:
                print(f"\n  {level.upper()} sample (first 3 rows):")
                # Show subset of columns for readability
                sample_cols = ['date', 'aggregation_level', 'price_exaa_mean', 'carbonprices_primary_market']
                if level == 'monthly':
                    sample_cols.extend(['climate_hdd_at', 'prod_gross_electricity_production', 'oegpi_month'])
                display_cols = [col for col in sample_cols if col in level_df.columns]
                print(level_df[display_cols].head(3).to_string(index=False))
        
        print()
        print("="*70)
        print("DIAGNOSTICS COMPLETE")
        print("="*70)
        
        return df
        
    except Exception as e:
        print(f"ERROR during diagnostics: {e}")
        return None

# Execute comprehensive diagnostics
consolidated_file = PROCESSED_DATA_PATH / "data_consolidated.csv"
final_diagnostics_df = run_comprehensive_diagnostics(consolidated_file)

if final_diagnostics_df is not None:
    print("\n✓ data_consolidated.csv is ready for EDA and modeling")
    print(f"  Contains: Electricity + Carbon + Climate + Production + Gas")
    print(f"  Total: {final_diagnostics_df.shape[0]} rows × {final_diagnostics_df.shape[1]} columns")
else:
    print("\n✗ Diagnostics failed - check errors above")

print("Cell 8: Comprehensive Diagnostics - COMPLETE")

COMPREHENSIVE DIAGNOSTICS: data_consolidated.csv

1. OVERALL STRUCTURE:
----------------------------------------------------------------------
  Shape: 4651 rows × 41 columns
  Date range (overall): 2010-01-01 to 2025-09-01
  Memory usage: 2.10 MB

2. AGGREGATION LEVEL DISTRIBUTION:
----------------------------------------------------------------------
  daily     :  3896 rows ( 83.8%) | Range: 2015-01-01 to 2025-08-31
  monthly   :   189 rows (  4.1%) | Range: 2010-01-01 to 2025-09-01
  weekly    :   566 rows ( 12.2%) | Range: 2015-01-05 to 2025-09-01

3. COLUMN INVENTORY BY DATA SOURCE:
----------------------------------------------------------------------
  Base structure: 7 columns
  Electricity:    5 columns - ['price_exaa_mean', 'price_mc_auction_mean', 'price_count_exaa', 'price_count_mc', 'data_completeness']
  Carbon:         3 columns - ['carbonprices_exchange_rate_eur_usd', 'carbonprices_primary_market', 'carbonprices_secondary_market']
  Climate:        4 columns - ['climat

In [4]:
import pandas as pd
from pathlib import Path

# Zeige aktuelles Verzeichnis
print(f"Current directory: {Path.cwd()}")

# Versuche verschiedene Pfade
possible_paths = [
    Path('data/processed/data_consolidated.csv'),
    Path('../data/processed/data_consolidated.csv'),
    Path('../../data/processed/data_consolidated.csv'),
    Path.cwd().parent / 'data' / 'processed' / 'data_consolidated.csv'
]

for path in possible_paths:
    if path.exists():
        print(f"✓ Found at: {path}")
        df = pd.read_csv(path)
        break
else:
    print("✗ File not found in any expected location")
    print("\nTry:")
    print("  import os")
    print("  os.chdir('C:/Users/paulr/Desktop/Capstone')")

Current directory: c:\Users\paulr\Desktop\Capstone\notebooks
✓ Found at: ..\data\processed\data_consolidated.csv


In [6]:
import pandas as pd

df = pd.read_csv('../data/processed/data_consolidated.csv')

# Prüfe die ersten 20 Zeilen der Gas-Columns
print(df[['date', 'aggregation_level', 'oegpi_month', 'oegpi_quarter']].head(20))

# Prüfe eine Zeile wo Gas-Daten sein SOLLTEN (ab 2020)
print("\n2020-01 Row (sollte Gas-Daten haben):")
print(df[df['date'] == '2020-01-01'][['date', 'oegpi_month', 'oegpi_quarter', 'oegpi_season', 'oegpi_year']])

# Öffne das CSV in einem Text-Editor und zeig mir die erste Zeile mit Gas-Daten
with open('../data/processed/data_consolidated.csv', 'r') as f:
    lines = f.readlines()
    # Zeige Header
    print("\nCSV Header:")
    print(lines[0][:200])  # Erste 200 chars
    
    # Finde Zeile mit 2020-01-01
    for i, line in enumerate(lines):
        if '2020-01-01' in line and 'monthly' in line:
            print(f"\n2020-01-01 row (line {i}):")
            print(line)
            break

          date aggregation_level  oegpi_month  oegpi_quarter
0   2010-01-01           monthly          NaN            NaN
1   2010-02-01           monthly          NaN            NaN
2   2010-03-01           monthly          NaN            NaN
3   2010-04-01           monthly          NaN            NaN
4   2010-05-01           monthly          NaN            NaN
5   2010-06-01           monthly          NaN            NaN
6   2010-07-01           monthly          NaN            NaN
7   2010-08-01           monthly          NaN            NaN
8   2010-09-01           monthly          NaN            NaN
9   2010-10-01           monthly          NaN            NaN
10  2010-11-01           monthly          NaN            NaN
11  2010-12-01           monthly          NaN            NaN
12  2011-01-01           monthly          NaN            NaN
13  2011-02-01           monthly          NaN            NaN
14  2011-03-01           monthly          NaN            NaN
15  2011-04-01          